# 3차시 실습 노트북
## 비선형 탐색 — 깊이 우선 탐색(DFS) · 너비 우선 탐색(BFS)
세종과학고등학교 정보과학 · 1학년

---

지난 시간까지는 **한 줄로 늘어선** 데이터를 순차 탐색·이분 탐색으로 훑었다.
오늘은 **여러 갈래로 뻗어 있는** 트리를 훑는다.

두 방법의 차이는 딱 하나다 — **대기열에서 무엇을 먼저 꺼내는가.**

| | 꺼내는 위치 | 자료 구조 |
|---|---|---|
| **DFS** | 맨 뒤 (LIFO) | 스택 · 재귀 |
| **BFS** | 맨 앞 (FIFO) | 큐 (deque) |

**오늘의 순서**
1~4번에서 두 알고리즘을 직접 작성하고, 5·6번에서 그것으로만 풀 수 있는 문제를 하나씩 푼다.

## 0. 오늘 쓸 트리 (교과서 p.178)

```
              a
        ┌─────┴─────┐
        b           c
     ┌──┴──┐     ┌──┴──┐
     d     e     f     g
     │   ┌─┴─┐   │
     h   i   j   k
```

각 노드의 **자식 목록**을 딕셔너리로 적는다. 자식이 없는 단말 노드는 **빈 리스트** `[]` 다.

아래 셀은 먼저 그냥 실행하자. 1~4번 문제에서 이 `tree`를 그대로 쓴다.

**실행 결과**: `노드 11 개 / 간선 10 개`

In [ ]:
tree = {
    'a': ['b', 'c'],
    'b': ['d', 'e'],
    'c': ['f', 'g'],
    'd': ['h'],
    'e': ['i', 'j'],
    'f': ['k'],
    'g': [],
    'h': [],
    'i': [],
    'j': [],
    'k': []
}

edge_count = 0
for node in tree:
    edge_count = edge_count + len(tree[node])

print('노드', len(tree), '개 / 간선', edge_count, '개')

## 1. DFS 재귀로 완성하기

루트 `a` 에서 출발해 **최대한 깊이** 내려가고, 막히면 되돌아온다(**백트랙**).

되돌아오는 동작은 따로 적을 필요가 없다. `dfs(child)` 가 끝나고 `for` 문의 다음 반복으로 넘어가는 것이 곧 백트랙이다.

**실행 결과**: `a -> b -> d -> h -> e -> i -> j -> c -> f -> k -> g`

In [ ]:
visited = []

def dfs(node):
    # ① 현재 노드를 방문 기록에 추가
    visited.append(node)

    # ② 자식을 순서대로 재귀 호출 → 더 깊이 내려간다
    for child in tree[node]:
        dfs(child)

    # ③ for 문이 끝나면 = 더 갈 곳이 없음 → 백트랙

dfs('a')
print(' -> '.join(visited))

## 2. DFS를 명시적 스택으로 완성하기

재귀 대신 **스택 리스트**를 직접 만들어도 결과는 같다.
파이썬 리스트에서 `pop()` 은 **맨 뒤**를 꺼낸다(LIFO).

한 가지 함정: 자식을 `b`, `c` 순으로 넣으면 `c` 가 먼저 꺼내진다.
1번과 같은 순서를 얻으려면 **역순으로 push** 해야 한다.

**실행 결과**: `['a', 'b', 'd', 'h', 'e', 'i', 'j', 'c', 'f', 'k', 'g']`

In [ ]:
visited = []
stack = ['a']          # 루트를 넣고 시작

while stack:
    # ① top을 꺼낸다 (LIFO)
    node = stack.pop()

    # ② 방문 기록
    visited.append(node)

    # ③ 자식을 역순으로 push
    for child in reversed(tree[node]):
        stack.append(child)

print(visited)

## 3. BFS를 큐로 완성하기

이번에는 **맨 앞**에서 꺼낸다. `deque` 의 `popleft()` 를 쓴다.

2번 코드와 비교해 보면 **꺼내는 쪽만** 바뀌었다. 넣는 방식은 똑같다.

**실행 결과**: `['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k']`

In [ ]:
from collections import deque

visited = []
q = deque(['a'])       # 큐에 루트 넣기

while q:
    # ① front를 꺼낸다 (FIFO)
    node = q.popleft()

    # ② 방문 기록
    visited.append(node)

    # ③ 자식을 큐 뒤에 넣는다
    for child in tree[node]:
        q.append(child)

print(visited)

## 4. BFS로 레벨과 경로 구하기

BFS는 레벨이 얕은 순서로 훑는다. 그래서 어떤 노드를 **처음 만나는 순간**이 곧 그 노드의 레벨(= 루트로부터의 최소 이동 횟수)이다.

`level` 에 깊이를, `parent` 에 "어디서 왔는지"를 함께 기록하면 **경로까지** 복원할 수 있다.

**실행 결과**:
```
{'a': 0, 'b': 1, 'c': 1, 'd': 2, 'e': 2, 'f': 2, 'g': 2, 'h': 3, 'i': 3, 'j': 3, 'k': 3}
a -> j : 레벨 3 , 경로 ['a', 'b', 'e', 'j']
```

In [ ]:
from collections import deque

def bfs_path(root, goal):
    level = {}
    level[root] = 0
    parent = {}
    parent[root] = None
    q = deque([root])

    while q:
        node = q.popleft()
        for child in tree[node]:
            # ① 자식의 레벨은 부모보다 한 칸 깊다
            level[child] = level[node] + 1
            # ② 어디서 왔는지 기록
            parent[child] = node
            q.append(child)

    # ③ 목표에서 거꾸로 되짚어 경로 복원
    path = []
    cur = goal
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    path.reverse()

    return level, level[goal], path

level, depth, path = bfs_path('a', 'j')
print(level)
print('a -> j : 레벨', depth, ', 경로', path)

## 5. 【DFS 문제】 합이 target이 되는 조합 찾기

> 정수 목록 `data` 에서 **몇 개를 골라** 합이 정확히 `target` 이 되게 할 수 있는가?
> 가능한 조합을 **모두** 찾아 출력하시오.

**주어진 데이터**: `data = [3, 34, 4, 12, 5, 2]`, `target = 9`

**실행 결과**:
```
[3, 4, 2]
[4, 5]
찾은 조합 수 : 2
```

### 왜 DFS 문제인가

각 수마다 **"고른다 / 고르지 않는다"** 두 갈래로 상태가 갈라진다.
수가 6개면 갈래가 6번 갈라지므로, 상태 공간은 아래 같은 **트리**가 된다.

```
                    (아무것도 안 고름)
              ┌────────────┴────────────┐
          3을 고름                  3을 안 고름
        ┌─────┴─────┐            ┌─────┴─────┐
    34 고름    34 안 고름     34 고름    34 안 고름
      ...          ...          ...          ...
```

이 트리를 **깊이 우선**으로 내려가며, 합이 `target` 이 되는 지점을 찾으면 된다.

앞에서 트리를 두 가지 방법으로 훑었듯이, 이 문제도 **재귀(5-1)** 와 **명시적 스택(5-2)** 두 가지로 풀어본다.
두 코드의 출력은 완전히 같다.

---

### 5-1. 재귀로 풀기 — 1번 코드와 같은 뼈대

1번의 `dfs(node)` 와 구조가 같다. 자식이 `tree[node]` 대신 **"고른다 / 안 고른다"** 두 개일 뿐이다.

들고 다녀야 할 정보가 세 가지라 매개변수가 셋이다.
- `i` : 이제 몇 번째 수를 볼 차례인가
- `total` : 지금까지 고른 수의 합
- `picked` : 지금까지 고른 수들

> **`picked.pop()` 이 바로 백트랙이다.**
> `picked` 하나를 계속 고쳐 쓰기 때문에, 한 갈래를 다 보고 나면 **원래 상태로 되돌려 놓아야** 다른 갈래를 제대로 탐색할 수 있다.

In [ ]:
data = [3, 34, 4, 12, 5, 2]
target = 9

found = []

def dfs(i, total, picked):
    # 합이 딱 맞으면 정답 하나 발견
    if total == target:
        found.append(list(picked))
        return

    # 합이 넘었으면 더 볼 필요 없다 (가지치기)
    if total > target:
        return

    # 더 볼 수가 없으면 끝
    if i == len(data):
        return

    # ① data[i] 를 고르는 경우
    picked.append(data[i])
    dfs(i + 1, total + data[i], picked)
    picked.pop()                      # ← 백트랙: 원래대로 되돌린다

    # ② data[i] 를 고르지 않는 경우
    dfs(i + 1, total, picked)

dfs(0, 0, [])

for combo in found:
    print(combo)
print('찾은 조합 수 :', len(found))

### 5-2. 명시적 스택으로 풀기 — 2번 코드와 같은 뼈대

같은 문제를 재귀 없이 풀어보자. 2번에서 만든 스택 코드가 그대로 쓰인다.
달라지는 것은 **스택에 무엇을 넣느냐**뿐이다.

| | 2번 (트리) | 5-2 (조합) |
|---|---|---|
| 스택에 넣는 것 | 노드 이름 `'b'` | 상태 3개 `(i, total, picked)` |
| 자식 | `tree[node]` | **고른다 / 안 고른다** 두 개 |
| 꺼내기 | `stack.pop()` | `stack.pop()` |

> **push 순서에 주의** — 2번에서 `reversed` 를 썼던 것과 같은 이유다.
> 스택은 **나중에 넣은 것이 먼저** 나오므로, "고르는 경우"를 먼저 탐색하려면 **"안 고르는 경우"를 먼저 push** 해야 한다.

> **여기서는 `picked.pop()` 이 없다.**
> 상태마다 `picked + [data[i]]` 로 **새 리스트**를 만들어 스택에 넣기 때문에, 되돌릴 것이 없다.
> 되돌아가는 일은 `stack.pop()` 이 알아서 해 준다 — 스택에 남아 있던 다음 상태를 꺼내는 것이 곧 백트랙이다.

**실행 결과**: 5-1과 **완전히 같다.**
```
[3, 4, 2]
[4, 5]
찾은 조합 수 : 2
```

In [ ]:
data = [3, 34, 4, 12, 5, 2]
target = 9

found = []

# 스택에 (볼 차례, 지금까지의 합, 지금까지 고른 수들) 을 넣는다
stack = [(0, 0, [])]

while stack:
    i, total, picked = stack.pop()

    # 합이 딱 맞으면 정답 하나 발견
    if total == target:
        found.append(picked)
        continue

    # 합이 넘었으면 더 볼 필요 없다 (가지치기)
    if total > target:
        continue

    # 더 볼 수가 없으면 끝
    if i == len(data):
        continue

    # ① data[i] 를 고르지 않는 경우 (나중에 꺼내지도록 먼저 push)
    stack.append((i + 1, total, picked))

    # ② data[i] 를 고르는 경우 (먼저 꺼내지도록 나중에 push)
    stack.append((i + 1, total + data[i], picked + [data[i]]))

for combo in found:
    print(combo)
print('찾은 조합 수 :', len(found))

## 6. 【BFS 문제】 최소 연산 횟수로 수 만들기

> 수 `1` 에서 시작한다. 한 번에 다음 두 가지 중 하나만 할 수 있다.
> - **`+1`** : 1을 더한다
> - **`×2`** : 2를 곱한다
>
> `27` 을 만들려면 **최소 몇 번**의 연산이 필요한가? 그때 만들어지는 수의 순서도 함께 출력하시오.

**실행 결과**:
```
최소 연산 횟수 = 7
만드는 순서 : [1, 2, 3, 6, 12, 13, 26, 27]
```

확인해 보자 — `1 →(×2) 2 →(+1) 3 →(×2) 6 →(×2) 12 →(+1) 13 →(×2) 26 →(+1) 27`. 정확히 7번이다.

### 왜 BFS 문제인가

지금 만든 수 하나가 곧 **상태**다. 그 상태에서 갈 수 있는 곳(= 자식)은 **`+1` 한 수**와 **`×2` 한 수** 두 개다.
그림으로 그리면 3번에서 훑은 트리와 똑같이 생겼다.

```
                        1
                  ┌─────┴─────┐
                +1│           │×2
                  2           2      ← 둘 다 2 (같은 수!)
            ┌─────┴─────┐
          3               4
      ┌───┴───┐       ┌───┴───┐
      4       6       5       8
     ...     ...     ...     ...
```

**BFS는 연산을 1번 한 수를 전부 → 2번 한 수를 전부 → …** 순서로 훑는다.
그래서 `27` 을 **처음 만나는 순간**이 곧 최소 연산 횟수다. 4번에서 구한 `level` 과 완전히 같은 원리다.

DFS로 하면 `1 → 2 → 3 → 4 → 5 → …` 로 `+1` 만 하며 한없이 내려갈 수 있다. 답을 찾더라도 그게 최소라는 보장이 없다.

### 트리와 딱 한 가지가 다르다

위 그림에서 `1+1` 과 `1×2` 가 **똑같이 2**가 된다. 트리에서는 없던 일이다.
서로 다른 길이 같은 수에 도달할 수 있으므로, **이미 만든 수인지 확인**하지 않으면 같은 수를 몇 번이고 큐에 넣게 된다.

그래서 `count` 딕셔너리가 두 가지 일을 동시에 한다 — **최소 횟수 기록** + **이미 만들었는지 표시**.

In [ ]:
from collections import deque

start = 1
goal = 27

count = {}
count[start] = 0
parent = {}
parent[start] = None
q = deque([start])

while q:
    now = q.popleft()

    if now == goal:
        break

    # 여기서 갈 수 있는 곳은 두 개다
    neighbors = [now + 1, now * 2]

    for nxt in neighbors:
        # 목표를 넘어가면 더 볼 필요 없다
        if nxt > goal:
            continue

        # ① 이미 만든 수라면 건너뛴다
        if nxt in count:
            continue

        # ② 처음 만든 수다 → 횟수와 출처를 적고 큐에 넣는다
        count[nxt] = count[now] + 1
        parent[nxt] = now
        q.append(nxt)

# 목표에서 거꾸로 되짚어 순서 복원 (4번과 같은 방법)
path = []
cur = goal
while cur is not None:
    path.append(cur)
    cur = parent[cur]
path.reverse()

print('최소 연산 횟수 =', count[goal])
print('만드는 순서 :', path)

---
수고했습니다!

**오늘의 한 문장** — 두 알고리즘의 뼈대는 같다.
> "대기열에서 하나 꺼내 → 이웃을 확인 → 다시 넣는다"

달라지는 것은 **어느 쪽에서 꺼내는가**뿐이다. 뒤에서 꺼내면 DFS, 앞에서 꺼내면 BFS.

| | 5번 문제 | 6번 문제 |
|---|---|---|
| 묻는 것 | 조건에 맞는 **경우가 몇 가지**인가 | **몇 번 만에** 갈 수 있는가 |
| 쓰는 것 | **DFS** | **BFS** |

다음 시간에는 오늘의 DFS 위에 **백트래킹**을 얹어, 가망 없는 가지를 잘라내며 탐색하는 방법을 배웁니다.
5번의 `if total > target: continue` 가 바로 그 가지치기의 맛보기였습니다.